# Python practice 11: concurrency: threads, processes and asyncio

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-11-concurrency-and-async.ipynb)

13 exercises on thread pools, locks, queues, asyncio, semaphores, timeouts and rate limiting. Each exercise names the [reference entries](../learning/python-reference-11-concurrency-and-async.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 5 easy, 7 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Parallel map that keeps order](#exercise-1-parallel-map-that-keeps-order) | Easy | [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor) |
| 2 | [Per-job errors with as_completed](#exercise-2-per-job-errors-with-as_completed) | Medium | [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor) |
| 3 | [A thread-safe total](#exercise-3-a-thread-safe-total) | Easy | [Locks and race conditions](../learning/python-reference-11-concurrency-and-async.ipynb#locks-and-race-conditions) |
| 4 | [Producer and consumers](#exercise-4-producer-and-consumers) | Medium | [Producer and consumer with `queue.Queue`](../learning/python-reference-11-concurrency-and-async.ipynb#producer-and-consumer-with-queuequeue) |
| 5 | [Split work into balanced chunks](#exercise-5-split-work-into-balanced-chunks) | Easy | [`ProcessPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#processpoolexecutor) |
| 6 | [Gather results concurrently](#exercise-6-gather-results-concurrently) | Easy | [Coroutines, `await` and `gather`](../learning/python-reference-11-concurrency-and-async.ipynb#coroutines-await-and-gather) |
| 7 | [Bounded concurrency with a semaphore](#exercise-7-bounded-concurrency-with-a-semaphore) | Medium | [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup) |
| 8 | [Timeout with a fallback](#exercise-8-timeout-with-a-fallback) | Easy | [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup) |
| 9 | [All-or-nothing with TaskGroup](#exercise-9-all-or-nothing-with-taskgroup) | Medium | [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup) |
| 10 | [Async worker pool](#exercise-10-async-worker-pool) | Medium | [Worker pools with `asyncio.Queue`](../learning/python-reference-11-concurrency-and-async.ipynb#worker-pools-with-asyncioqueue) |
| 11 | [Blocking calls off the event loop](#exercise-11-blocking-calls-off-the-event-loop) | Medium | [Coroutines, `await` and `gather`](../learning/python-reference-11-concurrency-and-async.ipynb#coroutines-await-and-gather) |
| 12 | [Token-bucket rate limiter](#exercise-12-token-bucket-rate-limiter) | Hard | [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup) |
| 13 | [First result wins](#exercise-13-first-result-wins) | Medium | [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction


import asyncio
import threading
import time
from concurrent.futures import ThreadPoolExecutor


def run(coro):
    """Run a coroutine to completion on a fresh event loop (works inside Jupyter's running loop too)."""
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()


def caught(fn):
    """Call fn and return the exception it raises, or None."""
    try:
        fn()
    except BaseException as error:  # noqa: BLE001
        return error
    return None


class Probe:
    """An async function that records how many calls run at the same time."""

    def __init__(self, delay=0.01):
        self.delay, self.active, self.peak, self.calls = delay, 0, 0, 0

    async def call(self, x):
        self.active += 1
        self.calls += 1
        self.peak = max(self.peak, self.active)
        await asyncio.sleep(self.delay)
        self.active -= 1
        return x * 2


async def sleepy(delay, value):
    await asyncio.sleep(delay)
    return value

## Exercise 1: Parallel map that keeps order

**Difficulty:** Easy · about 5 min · **Practises:** [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor)

*Fetching many objects from S3 or an API concurrently.*

Write `parallel_map(fn, items, workers=4)` returning `[fn(x) for x in items]` in input order, computed with a `ThreadPoolExecutor`.

**Sample calls**

| call | expected |
| --- | --- |
| `parallel_map(lambda x: x * x, range(6))` | `[0, 1, 4, 9, 16, 25]` |
| `parallel_map(str, [])` | `[]` |
| `parallel_map(str.upper, ["a", "b"])` | `['A', 'B']` |

In [ ]:
def parallel_map(fn, items, workers=4):
    # your code here
    pass

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def parallel_map(fn, items, workers=4):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(fn, items))

## Exercise 2: Per-job errors with as_completed

**Difficulty:** Medium · about 10 min · **Practises:** [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor)

*A batch of independent jobs should report every failure, not stop at the first.*

Write `run_jobs(jobs, workers=4)` taking `{name: fn}` and returning `(results, errors)`: dicts of name → return value and name → `"<ExceptionType>: <message>"`. Both dicts sorted by name.

**Sample calls**

| call | expected |
| --- | --- |
| `run_jobs({"b": lambda: 2, "a": lambda: 1 / 0, "c": lambda: "ok"})` | `({'b': 2, 'c': 'ok'}, {'a': 'ZeroDivisionError: division by zero'})` |
| `run_jobs({"x": lambda: 1})` | `({'x': 1}, {})` |
| `run_jobs({})` | `({}, {})` |

In [ ]:
def run_jobs(jobs, workers=4):
    # your code here
    pass

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_jobs(jobs, workers=4):
    results, errors = {}, {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(fn): name for name, fn in jobs.items()}
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as error:
                errors[name] = f"{type(error).__name__}: {error}"
    return dict(sorted(results.items())), dict(sorted(errors.items()))

## Exercise 3: A thread-safe total

**Difficulty:** Easy · about 5 min · **Practises:** [Locks and race conditions](../learning/python-reference-11-concurrency-and-async.ipynb#locks-and-race-conditions)

*Shared counters updated from worker threads lose updates without a lock.*

Write `locked_total(threads, increments)` starting `threads` threads that each add 1 to a shared total `increments` times under a `threading.Lock`, and returning the final total.

**Sample calls**

| call | expected |
| --- | --- |
| `locked_total(4, 2500)` | `10000` |
| `locked_total(1, 10)` | `10` |
| `locked_total(0, 100)` | `0` |

In [ ]:
def locked_total(threads, increments):
    # your code here
    pass

In [ ]:
import threading

def locked_total(threads, increments):
    total = 0
    lock = threading.Lock()

    def work():
        nonlocal total
        for _ in range(increments):
            with lock:
                total += 1

    workers = [threading.Thread(target=work) for _ in range(threads)]
    for w in workers:
        w.start()
    for w in workers:
        w.join()
    return total

## Exercise 4: Producer and consumers

**Difficulty:** Medium · about 15 min · **Practises:** [Producer and consumer with `queue.Queue`](../learning/python-reference-11-concurrency-and-async.ipynb#producer-and-consumer-with-queuequeue)

*Bounded queues between stages keep memory flat in streaming pipelines.*

Write `sum_squares_queue(items, consumers=3)` putting each item on a `queue.Queue(maxsize=5)`, with `consumers` threads squaring items and adding them to a shared total (under a lock), stopped by one sentinel per consumer. Return the total.

**Sample calls**

| call | expected |
| --- | --- |
| `sum_squares_queue([1, 2, 3])` | `14` |
| `sum_squares_queue(range(100))` | `328350` |
| `sum_squares_queue([])` | `0` |

In [ ]:
def sum_squares_queue(items, consumers=3):
    # your code here
    pass

In [ ]:
import queue
import threading

def sum_squares_queue(items, consumers=3):
    q, stop = queue.Queue(maxsize=5), object()
    total, lock = 0, threading.Lock()

    def consume():
        nonlocal total
        while (item := q.get()) is not stop:
            with lock:
                total += item * item

    threads = [threading.Thread(target=consume) for _ in range(consumers)]
    for t in threads:
        t.start()
    for item in items:
        q.put(item)
    for _ in threads:
        q.put(stop)
    for t in threads:
        t.join()
    return total

## Exercise 5: Split work into balanced chunks

**Difficulty:** Easy · about 5 min · **Practises:** [`ProcessPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#processpoolexecutor)

*Dividing a range across process workers so each gets a similar amount.*

Write `chunk_ranges(n, k)` splitting `range(n)` into `k` contiguous `(start, stop)` pairs whose sizes differ by at most one (larger chunks first), dropping empty chunks.

**Sample calls**

| call | expected |
| --- | --- |
| `chunk_ranges(10, 2)` | `[(0, 5), (5, 10)]` |
| `chunk_ranges(10, 3)` | `[(0, 4), (4, 7), (7, 10)]` |
| `chunk_ranges(2, 4)` | `[(0, 1), (1, 2)]` |

In [ ]:
def chunk_ranges(n, k):
    # your code here
    pass

In [ ]:
def chunk_ranges(n, k):
    size, extra = divmod(n, k)
    out, start = [], 0
    for i in range(k):
        stop = start + size + (1 if i < extra else 0)
        if stop > start:
            out.append((start, stop))
        start = stop
    return out

## Exercise 6: Gather results concurrently

**Difficulty:** Easy · about 5 min · **Practises:** [Coroutines, `await` and `gather`](../learning/python-reference-11-concurrency-and-async.ipynb#coroutines-await-and-gather)

*Calling an async API for many ids at once.*

Write `async def fetch_all(ids, fetch)` awaiting `fetch(id)` for every id concurrently and returning the results in input order. Checks run it with `run(...)`.

**Sample calls**

| call | expected |
| --- | --- |
| `run(fetch_all([3, 1, 2], lambda i: sleepy(0.01 * i, i * 10)))` | `[30, 10, 20]` |
| `run(fetch_all([], lambda i: sleepy(0, i)))` | `[]` |
| `run(fetch_all(["a"], lambda s: sleepy(0, s.upper())))` | `['A']` |

In [ ]:
async def fetch_all(ids, fetch):
    # your code here
    pass

In [ ]:
import asyncio

async def fetch_all(ids, fetch):
    return list(await asyncio.gather(*(fetch(i) for i in ids)))

## Exercise 7: Bounded concurrency with a semaphore

**Difficulty:** Medium · about 10 min · **Practises:** [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup)

*APIs throttle you if you send hundreds of requests at once.*

Write `async def map_limited(fn, items, limit)` awaiting `fn(x)` for all items with at most `limit` running at once (use `asyncio.Semaphore`), returning results in input order. The `Probe` helper records peak concurrency.

**Sample calls**

| call | expected |
| --- | --- |
| `run(map_limited(Probe().call, range(5), 2))` | `[0, 2, 4, 6, 8]` |
| `(lambda p: (run(map_limited(p.call, range(20), 3)), p.peak)[1])(Probe())` | `3` |
| `(lambda p: (run(map_limited(p.call, range(5), 1)), p.peak)[1])(Probe())` | `1` |

In [ ]:
async def map_limited(fn, items, limit):
    # your code here
    pass

In [ ]:
import asyncio

async def map_limited(fn, items, limit):
    sem = asyncio.Semaphore(limit)

    async def one(x):
        async with sem:
            return await fn(x)

    return list(await asyncio.gather(*(one(x) for x in items)))

## Exercise 8: Timeout with a fallback

**Difficulty:** Easy · about 5 min · **Practises:** [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup)

*Slow dependencies should degrade gracefully rather than hang a job.*

Write `async def with_timeout(coro, seconds, default=None)` returning the coroutine's result, or `default` if it takes longer than `seconds`.

**Sample calls**

| call | expected |
| --- | --- |
| `run(with_timeout(sleepy(0.01, "ok"), 1))` | `'ok'` |
| `run(with_timeout(sleepy(1, "late"), 0.05, "fallback"))` | `'fallback'` |
| `run(with_timeout(sleepy(1, 1), 0.05))` | `None` |

In [ ]:
async def with_timeout(coro, seconds, default=None):
    # your code here
    pass

In [ ]:
import asyncio

async def with_timeout(coro, seconds, default=None):
    try:
        async with asyncio.timeout(seconds):
            return await coro
    except TimeoutError:
        return default

## Exercise 9: All-or-nothing with TaskGroup

**Difficulty:** Medium · about 10 min · **Practises:** [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup)

*When one request fails the rest are wasted work; structured concurrency cancels them.*

Write `async def fetch_strict(ids, fetch)` running `fetch(id)` for all ids in an `asyncio.TaskGroup` and returning results in input order; if any fail, the `ExceptionGroup` propagates.

**Sample calls**

| call | expected |
| --- | --- |
| `run(fetch_strict([1, 2, 3], lambda i: sleepy(0.01, i * i)))` | `[1, 4, 9]` |
| `run(fetch_strict([], lambda i: sleepy(0, i)))` | `[]` |
| `type(caught(lambda: run(fetch_strict([1, 0], lambda i: sleepy(0, 1 / i))))).__name__` | `'ExceptionGroup'` |

In [ ]:
async def fetch_strict(ids, fetch):
    # your code here
    pass

In [ ]:
import asyncio

async def fetch_strict(ids, fetch):
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(fetch(i)) for i in ids]
    return [t.result() for t in tasks]

## Exercise 10: Async worker pool

**Difficulty:** Medium · about 15 min · **Practises:** [Worker pools with `asyncio.Queue`](../learning/python-reference-11-concurrency-and-async.ipynb#worker-pools-with-asyncioqueue)

*A fixed number of workers draining a queue: the standard async consumer shape.*

Write `async def process_all(items, workers, handle)` putting items on an `asyncio.Queue`, starting `workers` worker tasks that `await handle(item)` and collect results, waiting with `queue.join()`, cancelling the workers, and returning the sorted results.

**Sample calls**

| call | expected |
| --- | --- |
| `run(process_all([3, 1, 2], 2, Probe().call))` | `[2, 4, 6]` |
| `run(process_all([], 3, Probe().call))` | `[]` |
| `(lambda p: (run(process_all(range(12), 3, p.call)), p.peak)[1])(Probe())` | `3` |

In [ ]:
async def process_all(items, workers, handle):
    # your code here
    pass

In [ ]:
import asyncio

async def process_all(items, workers, handle):
    q, results = asyncio.Queue(), []
    for item in items:
        q.put_nowait(item)

    async def worker():
        while True:
            item = await q.get()
            try:
                results.append(await handle(item))
            finally:
                q.task_done()

    tasks = [asyncio.create_task(worker()) for _ in range(workers)]
    await q.join()
    for t in tasks:
        t.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)
    return sorted(results)

## Exercise 11: Blocking calls off the event loop

**Difficulty:** Medium · about 10 min · **Practises:** [Coroutines, `await` and `gather`](../learning/python-reference-11-concurrency-and-async.ipynb#coroutines-await-and-gather)

*Legacy blocking clients inside async code must run in threads.*

Write `async def run_blocking(funcs)` running each zero-argument blocking function with `asyncio.to_thread`, concurrently, and returning their results in order.

**Sample calls**

| call | expected |
| --- | --- |
| `run(run_blocking([lambda: 1, lambda: "two"]))` | `[1, 'two']` |
| `run(run_blocking([]))` | `[]` |
| `(lambda s: (run(run_blocking([lambda: time.sleep(0.2)] * 3)), time.perf_counter() - s < 0.5)[1])(time.perf_counter())` | `True` |

In [ ]:
async def run_blocking(funcs):
    # your code here
    pass

In [ ]:
import asyncio

async def run_blocking(funcs):
    return list(await asyncio.gather(*(asyncio.to_thread(f) for f in funcs)))

## Exercise 12: Token-bucket rate limiter

**Difficulty:** Hard · about 20 min · **Practises:** [Limiting concurrency, timeouts and `TaskGroup`](../learning/python-reference-11-concurrency-and-async.ipynb#limiting-concurrency-timeouts-and-taskgroup)

*Rate limiting is a common system-design coding question and a real need when calling APIs.*

Write class `TokenBucket(rate, capacity, clock=time.monotonic)` starting full; `allow()` refills `rate` tokens per second of elapsed clock time (up to `capacity`), then takes one token and returns True, or returns False if fewer than one token is available. Checks inject a fake clock.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda t: (lambda b: [b.allow() for _ in range(4)])(TokenBucket(1, 3, lambda: t[0])))([0.0])` | `[True, True, True, False]` |
| `(lambda t: (lambda b: [b.allow(), b.allow(), t.__setitem__(0, 1.0), b.allow(), b.allow()])(TokenBucket(1, 1, lambda: t[0])))([0.0])` | `[True, False, None, True, False]` |
| `(lambda t: (lambda b: [b.allow(), t.__setitem__(0, 100.0), sum(b.allow() for _ in range(5))][2])(TokenBucket(1, 2, lambda: t[0])))([0.0])` | `2` |

In [ ]:
class TokenBucket:
    # your code here
    pass

In [ ]:
import threading
import time

class TokenBucket:
    def __init__(self, rate, capacity, clock=time.monotonic):
        self.rate, self.capacity, self.clock = rate, capacity, clock
        self.tokens, self.updated = float(capacity), clock()
        self.lock = threading.Lock()

    def allow(self):
        with self.lock:
            now = self.clock()
            self.tokens = min(self.capacity, self.tokens + (now - self.updated) * self.rate)
            self.updated = now
            if self.tokens >= 1:
                self.tokens -= 1
                return True
            return False

## Exercise 13: First result wins

**Difficulty:** Medium · about 10 min · **Practises:** [`ThreadPoolExecutor`](../learning/python-reference-11-concurrency-and-async.ipynb#threadpoolexecutor)

*Hedged requests: ask several replicas and use whichever answers first.*

Write `first_result(funcs)` running zero-argument functions in a thread pool and returning the result of the first to finish successfully, using `wait(..., return_when=FIRST_COMPLETED)` in a loop; if all fail, raise the last error. Do not wait for slower functions once you have a result (`shutdown(wait=False, cancel_futures=True)`).

**Sample calls**

| call | expected |
| --- | --- |
| `(first_result([lambda: (time.sleep(0.3), "slow")[1], lambda: (time.sleep(0.01), "fast")[1]]))` | `'fast'` |
| `first_result([lambda: 1 / 0, lambda: (time.sleep(0.05), "ok")[1]])` | `'ok'` |
| `first_result([lambda: 42])` | `42` |

In [ ]:
def first_result(funcs):
    # your code here
    pass

In [ ]:
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

def first_result(funcs):
    pool = ThreadPoolExecutor(max_workers=max(1, len(funcs)))
    try:
        pending = {pool.submit(f) for f in funcs}
        error = None
        while pending:
            done, pending = wait(pending, return_when=FIRST_COMPLETED)
            for future in done:
                if future.exception() is None:
                    return future.result()
                error = future.exception()
        raise error if error else ValueError("no functions given")
    finally:
        pool.shutdown(wait=False, cancel_futures=True)

**Next:** look anything up in [Python reference 11: concurrency: threads, processes and asyncio](../learning/python-reference-11-concurrency-and-async.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).